# ShiftGuard-X — Notebook 48: Cross_Family_Research_Gate

**Purpose:** Independently execute candidate/ablation on 12 cross-family sklearn streams, optionally import NB46 and NB47 ZIPs, and emit a conservative research decision. These families have appeared in earlier SGX research and are NOT claimed to be unseen.

All code is standalone. Run **Runtime → Run all** in Google Colab (CPU). Each notebook writes a results ZIP to `/content` and attempts an automatic download. If external ALP dependencies or UCI downloads fail, it exports a diagnostic ZIP rather than inventing scores.

**Research integrity:** A native ALP estimated bits/value figure is *not* directly comparable to an encoded file containing its header; benchmarking alone does not establish novelty or Q1 journal readiness.


In [ ]:
import sys,os,io,csv,json,zipfile,time,hashlib,subprocess,shutil,struct,zlib,lzma,platform,traceback,statistics,urllib.request
from pathlib import Path
import numpy as np
ROOT=Path('/content') if Path('/content').exists() else Path('/mnt/data')
OUT=ROOT/'SGX_NB48_outputs';OUT.mkdir(parents=True,exist_ok=True)
TEST_MODE = os.environ.get('SGX_LOCAL_SMOKE_ONLY','0') == '1' # leave FALSE in Colab

def dump_json(name,x): (OUT/name).write_text(json.dumps(x,indent=2,default=str))
def dump_csv(name,rows):
    if not rows: return
    keys=list(dict.fromkeys(k for row in rows for k in row.keys()))
    with (OUT/name).open('w',newline='',encoding='utf8') as f:
        w=csv.DictWriter(f,fieldnames=keys);w.writeheader();w.writerows(rows)
def result_zip(no):
    path=ROOT/f'SGX_NB{no}_results.zip'
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        for f in sorted(OUT.rglob('*')):
            if f.is_file(): z.write(f,f.relative_to(OUT))
    print('RESULTS ZIP',path,'bytes',path.stat().st_size)
    try:
        from google.colab import files
        files.download(str(path))
    except ImportError: pass
    except Exception as e: print('Manual download:',path,e)
    return path
def run(args,timeout=360,cwd=None,env=None):
    t=time.monotonic()
    try:
        p=subprocess.run(args,cwd=cwd,env=env,text=True,capture_output=True,timeout=timeout,check=False)
        return {'cmd':args,'returncode':p.returncode,'seconds':round(time.monotonic()-t,2),'stdout_tail':p.stdout[-12000:],'stderr_tail':p.stderr[-12000:]}
    except Exception as e:return {'cmd':args,'returncode':-1,'seconds':round(time.monotonic()-t,2),'exception':repr(e)}
def sha(raw):return hashlib.sha256(raw).hexdigest()
def stream_bytes(a):return np.ascontiguousarray(a,dtype='<f8').tobytes()


In [ ]:
TAG={0:'zlib',1:'shuffle_zlib',2:'xor1_zlib',3:'xor2_zlib',4:'xor4_zlib',5:'xor8_zlib'}
BS=256

def do_xor(raw,lag):
    a=np.frombuffer(raw,dtype='<u8'); v=a.copy()
    if len(a)>lag: v[lag:]=a[lag:] ^ a[:-lag]
    return v.astype('<u8').tobytes()
def undo_xor(raw,lag):
    a=np.frombuffer(raw,dtype='<u8').copy()
    for i in range(lag,len(a)):a[i]^=a[i-lag]
    return a.astype('<u8').tobytes()
def shuf(raw):
    return np.frombuffer(raw,dtype=np.uint8).reshape(-1,8).T.copy().tobytes()
def unshuf(raw):
    return np.frombuffer(raw,dtype=np.uint8).reshape(8,-1).T.copy().tobytes()
def do_block(raw,tag):
    if tag==0:return zlib.compress(raw,6)
    if tag==1:return zlib.compress(shuf(raw),6)
    lag={2:1,3:2,4:4,5:8}[tag]
    return zlib.compress(do_xor(raw,lag),6)
def undo_block(blob,tag):
    raw=zlib.decompress(blob)
    if tag==0:return raw
    if tag==1:return unshuf(raw)
    return undo_xor(raw,{2:1,3:2,4:4,5:8}[tag])

def sgx_encode(raw,enabled_tags=(0,1,2,3,4,5),block_size=BS):
    if len(raw)%8:raise ValueError('Input not float64-aligned')
    n=len(raw)//8
    blocks=(n+block_size-1)//block_size
    out=bytearray(b'SGX8'+struct.pack('<QHI',n,block_size,blocks))
    modes=[]
    for i in range(blocks):
        b=raw[i*block_size*8:min((i+1)*block_size*8,len(raw))]
        variants=[(len(payload),tag,payload) for tag in enabled_tags for payload in [do_block(b,tag)]]
        _,tag,payload=min(variants,key=lambda r:(r[0],r[1]))
        out.extend(struct.pack('<BI',tag,len(payload)));out.extend(payload);modes.append(tag)
    return bytes(out),modes

def sgx_decode(blob):
    if len(blob)<18 or blob[:4]!=b'SGX8':raise ValueError('Bad header')
    n,bs,nblock=struct.unpack('<QHI',blob[4:18]);p=18
    if bs==0 or nblock!=(n+bs-1)//bs:raise ValueError('Bad block count')
    pieces=[]
    for i in range(nblock):
        if p+5>len(blob):raise ValueError('Missing block header')
        tag,size=struct.unpack('<BI',blob[p:p+5]);p+=5
        if tag not in TAG or p+size>len(blob):raise ValueError('Invalid block')
        raw=undo_block(blob[p:p+size],tag);p+=size
        expected=min(bs,n-i*bs)*8
        if len(raw)!=expected:raise ValueError('Bad block length')
        pieces.append(raw)
    if p!=len(blob):raise ValueError('Trailing encoded data')
    out=b''.join(pieces)
    if len(out)!=n*8:raise ValueError('Size mismatch')
    return out

def bench_encode_decode(name,raw,reps=3):
    measures=[];selection=[]
    for _ in range(reps):
        t=time.perf_counter_ns()
        if name=='sgx_adaptive': buf,selection=sgx_encode(raw)
        elif name=='sgx_zlib_only': buf,selection=sgx_encode(raw,(0,))
        elif name=='sgx_shuffle_only': buf,selection=sgx_encode(raw,(1,))
        elif name=='sgx_xor1_only': buf,selection=sgx_encode(raw,(2,))
        elif name=='zlib6':buf=zlib.compress(raw,6)
        elif name=='lzma1':buf=lzma.compress(raw,preset=1)
        elif name=='zstd3':
            import zstandard as zstd
            buf=zstd.ZstdCompressor(level=3).compress(raw)
        else:raise ValueError(name)
        enc_ms=(time.perf_counter_ns()-t)/1e6
        t=time.perf_counter_ns()
        if name.startswith('sgx_'):decoded=sgx_decode(buf)
        elif name=='zlib6':decoded=zlib.decompress(buf)
        elif name=='lzma1':decoded=lzma.decompress(buf)
        else:
            import zstandard as zstd
            decoded=zstd.ZstdDecompressor().decompress(buf)
        dec_ms=(time.perf_counter_ns()-t)/1e6
        assert decoded==raw,f'BIT-EXACT VIOLATION: {name}'
        measures.append((len(buf),enc_ms,dec_ms))
    return {'bytes':measures[0][0],'enc_ms':float(statistics.median(x[1] for x in measures)),'dec_ms':float(statistics.median(x[2] for x in measures)),'mode_counts':{TAG[t]:selection.count(t) for t in sorted(set(selection))} if name=='sgx_adaptive' else {}}

def edge_tests():
    bits=np.array([0,0x8000000000000000,0x7ff0000000000000,0xfff0000000000000,0x7ff8000000000001,0x0000000000000001,0xffffffffffffffff],dtype='<u8')
    raw=bits.tobytes()*100
    for m in ['sgx_adaptive','sgx_zlib_only','sgx_shuffle_only','sgx_xor1_only']:
        assert bench_encode_decode(m,raw,1)['bytes']>0
    return 'PASS (0,-0,Inf,-Inf,NaN payload,subnormal,all-ones)'


In [ ]:
# NB48 runs cross-family follow-up even if NB46/NB47 ZIPs are absent.
from sklearn.datasets import load_digits,load_diabetes,load_breast_cancer,load_wine
sources=[('digits',load_digits().data,[2,10,50]),('diabetes',load_diabetes().data,[0,2,5]),('breast_cancer',load_breast_cancer().data,[0,5,20]),('wine_classification',load_wine().data,[0,4,8])]
status={'gate':'INCOMPLETE','inputs':{},'same_input_alp_native_bpv':'NOT_PROVEN','matched_full_container_alp':'NOT_AVAILABLE','novelty':'NOT_ESTABLISHED','dataset_novelty_note':'Cross-family benchmark only; these sklearn families have appeared in prior SGX notebooks','ground_truth_rule':'ALP native dry BPV is not complete compressed file size'}
for n in (46,47):
    p=ROOT/f'SGX_NB{n}_results.zip'
    if not p.exists() and not TEST_MODE:
        try:
            from google.colab import files
            print(f'Upload {p.name} now to integrate results, or cancel upload to continue with cross-family checks.')
            files.upload()
        except (ImportError,Exception) as e:print('Archive not supplied:',repr(e))
    if p.exists():
        try:
            with zipfile.ZipFile(p) as z:
                key='benchmark_status.json' if n==46 else 'status.json'
                status['inputs'][f'NB{n}']=json.loads(z.read(key));status['inputs'][f'NB{n}']['archive_present']=True
                for member in z.namelist():
                    if member in ('input_manifest.json','candidate_ablation.csv','alp_native_results.csv','generic_native_sizes.csv'):
                        (OUT/f'NB{n}_{member}').write_bytes(z.read(member))
        except Exception as e:status['inputs'][f'NB{n}']={'read_error':repr(e)}
    else:status['inputs'][f'NB{n}']={'archive_present':False}
edge=edge_tests(); rows=[];errors=[];metrics=[]
try:
    methods=['zlib6','lzma1','sgx_adaptive','sgx_zlib_only','sgx_shuffle_only','sgx_xor1_only']
    try:import zstandard;methods.append('zstd3')
    except ImportError:pass
    for dataset,data,columns in sources:
        for j in columns:
            raw=stream_bytes(data[:,j]);name=f'{dataset}__c{j:02d}'; digest=sha(raw)
            sizes={}
            for m in methods:
                try:
                    res=bench_encode_decode(m,raw,3)
                    sizes[m]=res['bytes']
                    rows.append({'dataset':dataset,'stream':name,'n':len(raw)//8,'sha256':digest,'mode':m,'bytes':res['bytes'],'bpv':8*res['bytes']/(len(raw)//8),'encode_ms':res['enc_ms'],'decode_ms':res['dec_ms'],'exact':True})
                except Exception as e:errors.append({'stream':name,'mode':m,'error':repr(e)})
            if 'sgx_adaptive' in sizes and 'zlib6' in sizes and 'lzma1' in sizes:
                best=min((sizes[m],m) for m in sizes if not m.startswith('sgx_'))
                metrics.append({'stream':name,'sgx_bytes':sizes['sgx_adaptive'],'best_generic_bytes':best[0],'best_generic_name':best[1],'saving_pct':100*(best[0]-sizes['sgx_adaptive'])/best[0]})
    complete=(len(errors)==0 and len(metrics)==12)
    wins=sum(m['saving_pct']>0 for m in metrics);losses=sum(m['saving_pct']<0 for m in metrics)
    status.update({'cross_family_streams':len(metrics),'cross_family_wins':wins,'cross_family_losses':losses,'cross_family_zero':len(metrics)-wins-losses,'cross_family_bit_exact':complete,'special_bits':edge,'measurement_errors':errors})
    previous=status['inputs'].get('NB46',{})
    bench_good=bool(previous.get('alp_benchmark_ran',False))
    status['same_input_alp_native_bpv']='POSSIBLE_NATIVE_ONLY' if bench_good else 'NOT_PROVEN'
    # This is a conservative gate: no ALP full file, thus no strong competitive claim.
    status['gate']=('EXPLORATORY_SIGNAL_NOT_Q1_READY' if complete and wins>=6 and losses<=2 else ('RETHINK_PROTOTYPE' if complete else 'INCOMPLETE'))
except Exception as e:status.update({'gate':'ERROR','error':repr(e),'traceback':traceback.format_exc()[-5000:]})
dump_csv('cross_family_ablation.csv',rows);dump_csv('cross_family_stream_summary.csv',metrics);dump_json('research_gate.json',status)
print('NB48 DECISION',json.dumps(status,indent=2)[:8000]);result_zip(48)
